<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_10_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 10

Now we move to LU decomposition. LU, like QR, is one of the computational back‐
bones underlying data-science algorithms, including least squares model fitting and
the matrix inverse. This chapter is, therefore, pivotal to your linear algebra education.
The thing about LU decomposition is you cannot simply learn it immediately.
Instead, you first need to learn about systems of equations, row reduction, and
Gaussian elimination. And in the course of learning those topics, you’ll also learn
about echelon matrices and permutation matrices. Oh yes, dear reader, this will be an
exciting and action-packed chapter.
Systems of Equations
To understand LU decomposition and its applications, you need to understand row
reduction and Gaussian elimination. And to understand those topics, you need to
understand how to manipulate equations, convert them into a matrix equation, and
solve that matrix equation using row reduction.
Let’s start with a “system” of one equation:
2x = 8
As I’m sure you learned in school, you can do various mathematical manipulations to
the equation—as long as you do the same thing to both sides of the equation. That
means that the following equation is not the same as the previous one, but they are
related to each other by simple manipulations. More importantly, any solution to one
equation is a solution to the other:
5 2x − 3 = 5 8 − 3

159

Now let’s move to a system of two equations:
x = 4 − y
y = x/2 + 2
In this system of equations, it is impossible to solve for unique values of x and y
from either of those equations alone. Instead, you need to consider both equations
simultaneously to derive the solution. If you try to solve that system now, you would
probably take the strategy of substituting y in the first equation with the right-hand
side of the second equation. After solving for x in the first equation, you plug that
value into the second equation to solve for y. This strategy is similar to (though not as
efficient as) back substitution, which I’ll define later.
An important feature of a system of equations is that you can add or subtract
individual equations to each other. In the following equations, I’ve added two times
the second equation to the first, and subtracted the first original equation from the
second (parentheses added for clarity):
x + 2y = 4 − y + x + 4
y − x = x/2 + 2 − 4 − y
I will let you work through the arithmetic, but the upshot is that x drops out of the
first equation while y drops out of the second equation. That makes the solution
much easier to calculate (x = 4/3, y = 8/3). Here’s the important point: scalar multiply‐
ing equations and adding them to other equations made the solution to the system
easier to find. Again, the modulated and original systems are not the same equations,
but their solutions are the same because the two systems are linked by a series of
linear operations.
This is the background knowledge you need to learn how to solve systems of equa‐
tions using linear algebra. But before learning that approach, you need to learn how
to represet a system of equations using matrices and vectors.
Converting Equations into Matrices
Converting systems of equations into a matrix-vector equation is used to solve sys‐
tems of equations, and it’s used to set up the formula for the general linear model in
statistics. Fortunately, translating equations into matrices is conceptually simple, and
involves two steps.
First, organize the equations so that the constants are on the right-hand side of the
equations. The constants are the numbers that are unattached to the variables (some‐
times called intercepts or offsets). The variables and their multiplying coefficients are

160 | Chapter 10: Row Reduction and LU Decomposition

on the left-hand side of the equation, in the same order (e.g., all equations should
have the x term first, then the y term, and so on). The following equations form the
system of equations we’ve been working with, in the proper organization:
x + y = 4
−x/2 + y = 2
Second, separate the coefficients (the numbers multiplying the variables; variables
that are missing from an equation have a coefficient of zero) into a matrix with one
row per equation. The variables are placed into a column vector that right-multiplies
the coefficients matrix. And the constants are placed into a column vector on the
right-hand side of the equation. Our example system has a matrix equation that looks
like this:
1 1
−1/2 1
x
y
=
4
2

And voilà! You’ve converted a system of equations into one matrix equation. We can
refer to this equation as Ax = b, where A is the matrix of coefficients, x is a vector of
unknown variables to solve for (in this case, x is the vector comprising [x y]), and b is
a vector of constants.
Please take a moment to make sure you understand how the matrix equation maps
onto the system of equations. In particular, work through the matrix-vector multipli‐
cation to demonstrate that it equals the original system of equations.
Working with Matrix Equations
You can manipulate matrix equations just like normal equations, including adding,
multiplying, transposing, etc., as long as the manipulations are valid (e.g., matrix
sizes match for addition) and all manipulations affect both sides of the equation. For
example, the following progression of equations is valid:
Ax = b
v + Ax = v + b
v + Ax T
= v + b
T

The main difference between working with matrix equations versus scalar equations
is that because matrix multiplication is side-dependent, you must multiply matrices in
the same way on both sides of the equation.

Systems of Equations | 161

1 Of course you know that order matters, but empirical demonstrations help build intuition. And I want you to
get in the habit of using Python as a tool to empirically confirm principles in mathematics.
For example, the following progression of equations is valid:
AX = B
CAX = CB
Notice that C premultiplies both sides of the equation. In contrast, the following
progression is not valid:
AX = B
AXC = CB
The problem here is that C postmultiplies in the left-hand side but premultiplies
in the right-hand side. To be sure, there will be a few exceptional cases where that
equation is valid (e.g., if C is the identity or zeros matrix), but in general that
progression is not valid.
Let’s see an example in Python. We will solve for the unknown matrix X in the
equation AX = B. The following code generates A and B from random numbers. You
already know that we can solve for X by using A

−1. The question is whether the order

of multiplication matters.1

In [ ]:
A = np.random.randn(4,4)
B = np.random.randn(4,4)
# solve for X
X1 = np.linalg.inv(A) @ B
X2 = B @ np.linalg.inv(A)
# residual (should be zeros matrix)
res1 = A@X1 - B
res2 = A@X2 - B

Row reduction is a topic that gets a lot of attention in traditional linear algebra,
because it is the time-honored way to solve systems of equations by hand. I seriously
doubt you’ll solve any systems of equations by hand in your career as a data scientist.
But row reduction is useful to know about, and it leads directly to LU decomposition,
which actually is used in applied linear algebra. So let’s begin.
Row reduction means iteratively applying two operations—scalar multiplication and
addition—to rows of a matrix. Row reduction relies on the same principle as adding
equations to other equations within a system.
Memorize this statement: The goal of row reduction is to transform a dense matrix into
an upper-triangular matrix.
Let’s start with a simple example. In the following dense matrix, we add the first row
to the second row, which knocks out the −2. And with that, we’ve transformed our
dense matrix into an upper-triangular matrix:
2 3
−2 2
R1 + R2
2 3
0 5

The upper-triangular matrix that results from row reduction is called the echelon
form of the matrix.
Formally, a matrix is in echelon form if (1) the leftmost nonzero number in a row
(which is called the pivot) is to the right of the pivot of rows above, and (2) any rows
of all zeros are below rows containing nonzeros.
Similar to manipulating equations in a system, the matrix after row reduction is
different from the matrix before row reduction. But the two matrices are linked by a
linear transform. And because linear transforms can be represented by matrices, we
can use matrix multiplication to express row reduction:
1 0
1 1
2 3
−2 2
=
2 3
0 5

Row Reduction | 163

2 Spoiler alert: LU decomposition involves representing a matrix as the product of a lower-triangular and an
upper-triangular matrix.
I will call that matrix L

−1 for reasons that will become clear when I introduce LU

decomposition.2 Thus, in the expression L

−1A = U , L

−1 is the linear transformation
that keeps track of the manipulations we’ve implemented through row reduction. For
now, you don’t need to focus on L

−1—in fact, it’s often ignored during Gaussian elimi‐
nation. But the key point (slightly expanded from an earlier claim) is this: row reduction
involves transforming a matrix into an upper-triangular matrix via row manipulations,
which can be implemented as premultiplication by a transformation matrix.
Here’s another example of a 3 × 3 matrix. This matrix requires two steps to transform
into its echelon form:
1 2 2
−1 3 0
2 4 −3
−2R1 + R3

1 2 2
−1 3 0
0 0 −7
R1 + R2
1 2 2
0 5 2
0 0 −7

Row reduction is tedious (see “Is Row Reduction Always So Easy?”). Surely there
must be a Python function to do it for us! Well, there is and there isn’t. There is no
Python function that returns an echelon form like what I created in the two previous
examples. The reason is that the echelon form of a matrix is not unique. For example,
in the previous 3 × 3 matrix, you could multiply the second row by 2 to give a row
vector of [0 10 4]. That creates a perfectly valid—but different—echelon form of
the same original matrix. Indeed, there is an infinite number of echelon matrices
associated with that matrix.
That said, two echelon forms of a matrix are preferred over the infinite possible
echelon forms. Those two forms are unique given some constraints and are called the
reduced row echelon form and U from LU decomposition. I will introduce both later;
first, it’s time to learn how to use row reduction to solve systems of equations.

Is Row Reduction Always So Easy?

Row reducing a matrix to its echelon form is a skill that requires diligent practice
and learning several tricks with cool-sounding names like row swapping and partial
pivoting. Row reduction reveals several interesting properties of the row and column
spaces of the matrix. And although successfully row reducing a 5 × 6 matrix with row
swaps by hand brings a feeling of accomplishment and satisfaction, my opinion is that
your time is better spent on concepts in linear algebra that have more direct applica‐

164 | Chapter 10: Row Reduction and LU Decomposition

3 Please conjure into your imagination that Matrix meme with Morpheus proffering the red and blue pill,
corresponding to new knowledge versus sticking to what you know.
tion to data science. We are building toward understanding LU decomposition, and
this brief introduction to row reduction is sufficient for that goal.
Gaussian Elimination
At this point in the book, you know how to solve matrix equations using the matrix
inverse. What if I told you that you could solve a matrix equation without inverting
any matrices?3
This technique is called Gaussian elimination. Despite its name, the algorithm was
actually developed by Chinese mathematicians nearly two thousand years before
Gauss and then rediscovered by Newton hundreds of years before Gauss. But Gauss
made important contributions to the method, including the techniques that modern
computers implement.
Gaussian elimination is simple: augment the matrix of coefficients by the vector of
constants, row reduce to echelon form, and then use back substitution to solve for
each variable in turn.
Let’s start with the system of two equations that we solved earlier: